# Prep Qwen3.5 wheels for offline RTX 6000 Pro training

Kaggle's RTX 6000 Pro runtime can't use Internet, but the Blackwell-Triton
wheel bundle (`mayukh18/nemotron-packages`) ships `transformers==4.57.6`
and an old `unsloth` that don't know the `qwen3_5` architecture.

Run this notebook on a **CPU runtime with Internet ON** to download the
upgraded wheels, then **Save Version** and **Output → New Dataset** to
publish them as `qwen35-wheels`. The training notebook attaches that
dataset and installs from it with `--no-index --find-links`.

Kaggle CPU and GPU runtimes share the same Python 3.12 / manylinux
x86_64 image, so wheels downloaded here install cleanly on the GPU box.

## Steps

1. New Notebook → Accelerator **None (CPU)** → Internet **On**.
2. Upload this `.ipynb` (File → Import Notebook) and Run All.
3. Save Version → Save & Run All (Commit).
4. After commit, the notebook output folder becomes a new dataset; name it `qwen35-wheels`.
5. In `train_kaggle2.ipynb`, Add Data → Datasets → search your `qwen35-wheels` and attach it.

In [ ]:
import os, subprocess, sys

OUT = "/kaggle/working/qwen35-wheels"
os.makedirs(OUT, exist_ok=True)

# --no-deps so we don't drag in torch (~2 GB, already in nemotron-packages).
# We explicitly include the transitive deps that transformers 5.x typically
# bumps so the offline install on the GPU box has everything it needs.
pkgs = [
    "transformers>=5.2.0",
    "unsloth",
    "unsloth_zoo",
    "tokenizers",
    "safetensors",
    "huggingface_hub",
    "accelerate",
    "peft",
    "trl",
    "datasets",
]

subprocess.run(
    [sys.executable, "-m", "pip", "download",
     "--dest", OUT,
     "--no-cache-dir",
     "--no-deps",
     *pkgs],
    check=True,
)

files = sorted(os.listdir(OUT))
total_mb = sum(os.path.getsize(os.path.join(OUT, f)) for f in files) / 1024 / 1024
print(f"Downloaded {len(files)} wheels, {total_mb:.1f} MB total")
for f in files:
    sz = os.path.getsize(os.path.join(OUT, f)) / 1024 / 1024
    print(f"  {f}: {sz:.2f} MB")

In [ ]:
# Sanity check: try resolving the same pin set from the local dir only.
# If this dry-run install fails on this CPU box, it will also fail on the
# GPU box — better to catch it here while Internet is still on.
subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "--dry-run",
     "--no-index", "--find-links", OUT,
     "transformers>=5.2.0", "unsloth", "unsloth_zoo"],
    check=True,
)
print("Offline resolve OK — wheels are self-sufficient for transformers/unsloth.")

## After commit

- Save Version → Save & Run All (Commit). Wait for the run to finish.
- On the notebook page, **Output** tab → **New Dataset** → title `qwen35-wheels`.
- Open `train_kaggle2.ipynb`, Add Data → search `qwen35-wheels`, attach.
- In `train_kaggle2.ipynb` cell 6 the path is auto-discovered via glob, so no edits needed.